# Scikit-Learn Models

This notebook demonstrates how to train one of the Scikit-Learn models included in our SklearnModels wrapper using the 3W dataset for Multiclass Classification with Time Series data using the ThreeWToolkit.

In [ ]:
from ThreeWToolkit.trainer import SklearnTrainerConfig
from ThreeWToolkit.models import SklearnModelsConfig
from ThreeWToolkit.dataset import ParquetDatasetConfig, TransformConfig
from ThreeWToolkit.preprocessing import (
    SequentialPreprocessingAdapterConfig,
    FillLabelsConfig,
    CleanSignalsConfig,
    ImputeMissingConfig,
    NormalizeConfig,
    RemapClassConfig,
)
from ThreeWToolkit.feature_extraction import (
    SequentialFeatureAdapterConfig,
    WindowingConfig,
    ConcatFeatureAdapterConfig,
    StatisticalConfig,
    EWStatisticalConfig,
    WaveletConfig,
)
from ThreeWToolkit.assessment import (
    AssessmentVisualizationConfig,
    ModelAssessmentConfig,
)
import matplotlib.pyplot as plt

RANDOM_SEED = 2026

## Loading and preparing the Dataset
The next step is to create a ParqueDataset instance to interact with the 3W dataset, for that we have to define a path location where we want to save the Dataset to (or where it is already located).

In [ ]:
# Modify this path to the folder where your dataset is downloaded
dataset_path = "../../dataset"
event_types = ["real", "simulated", "drawn"]

ds_train = ParquetDatasetConfig(path=dataset_path, event_type=["real"]).build()
ds_val = ParquetDatasetConfig(path=dataset_path, event_type=["simulated"]).build()
ds_test = ParquetDatasetConfig(path=dataset_path, event_type=["drawn"]).build()

[ParquetDataset] Dataset found at ../../dataset
[ParquetDataset] Validating dataset integrity...
[ParquetDataset] Dataset integrity check passed!


{'signal':                      ABER-CKGL  ABER-CKP  ESTADO-DHSV  ESTADO-M1  ESTADO-M2  \
 timestamp                                                                     
 2017-04-24 22:01:56        0.0       0.0     0.867921   0.414652  -0.681653   
 2017-04-24 22:01:57        0.0       0.0     0.867921   0.414652  -0.681653   
 2017-04-24 22:01:58        0.0       0.0     0.867921   0.414652  -0.681653   
 2017-04-24 22:01:59        0.0       0.0     0.867921   0.414652  -0.681653   
 2017-04-24 22:02:00        0.0       0.0     0.867921   0.414652  -0.681653   
 ...                        ...       ...          ...        ...        ...   
 2017-04-25 03:59:56        0.0       0.0     0.867921   0.414652  -0.681653   
 2017-04-25 03:59:57        0.0       0.0     0.867921   0.414652  -0.681653   
 2017-04-25 03:59:58        0.0       0.0     0.867921   0.414652  -0.681653   
 2017-04-25 03:59:59        0.0       0.0     0.867921   0.414652  -0.681653   
 2017-04-25 04:00:00        0.

In [ ]:
window_size = 128
dataset_processor = TransformConfig(
    pre_processing=SequentialPreprocessingAdapterConfig(
        steps=[
            CleanSignalsConfig(missing_column_threshold=0.65),
            ImputeMissingConfig(),
            NormalizeConfig(),
            FillLabelsConfig(),
            RemapClassConfig(),
        ]
    ),
    feature_extraction=SequentialFeatureAdapterConfig(
        steps=[
            WindowingConfig(window_size=window_size),
            ConcatFeatureAdapterConfig(
                steps=[StatisticalConfig(), EWStatisticalConfig(), WaveletConfig()]
            ),
        ]
    ),
).build()

dataset_processor.fit(ds_train)
ds_train_transformed = dataset_processor.transform(ds_train)
ds_val_transformed = dataset_processor.transform(ds_val)
ds_test_transformed = dataset_processor.transform(ds_test)


## Model Trainer Configurations

With the data ready, we are now able to define the `ModelTrainer`, using using the SkLearnModelsConfig and TrainerConfig configuration classes.

Very important to note that the TrainerConfig class receives as a parameter the model configuration (config_model), and we use this information to instantiate `ModelTrainer`.

In [ ]:
# First, create the configuration for the specific scikit-learn model you want to use.
from sklearn.ensemble import RandomForestClassifier

sklearn_config = SklearnModelsConfig(
    model_type=RandomForestClassifier,
    model_params={"n_estimators": 100, "max_depth": 10},
)

trainer = SklearnTrainerConfig(
    config_model=sklearn_config,
).build()

print("ModelTrainer configured for RandomForestClassifier:")
print(trainer.model)

ModelTrainer configured for RandomForestClassifier:
RandomForestClassifier(max_depth=10, random_state=2025)


## Training the model

With the data and Trainer ready, we can call the trainer.train() method while passing the x_train and y_train argument.

In [ ]:
train_results = trainer.train(
    train_dataset=ds_train_transformed, val_dataset=ds_val_transformed
)


## Training Assessment and Results

For gathering the results using the test set, we will use the ModelAssessmentConfig inside the trainer.assess method.

In [ ]:
test_results = trainer.predict(ds_test_transformed)

# Evaluate model performance on validation set using ModelTrainer's test method
assessment = ModelAssessmentConfig(
    metrics=["accuracy"],
).build()

results = assessment.evaluate(training_results=train_results, predictions=test_results)


Results exported to /home/eduardo/3W/toolkit/output
Model Assessment Summary
Model: SklearnModels
Task Type: TaskTypeEnum.CLASSIFICATION
Timestamp: 2026-01-16T18:57:41.564163

Metrics:
  accuracy: 1.0000
Test Metrics: {'accuracy': 1.0}


In [ ]:
history = train_results.history
plt.figure(figsize=(10, 5))
plt.plot(history.val_loss, label="Val Loss")
plt.plot(history.train_loss, label="Train Loss")
plt.title("Training and Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()